#### Use LLM as second round of filtering after keywords match

In [24]:
import os,sys
sys.path.insert(0,'../libs')
sys.path.insert(0,'../src')
from utils import load_json
import openai
from llm_utils import BSAgent,custom_llm_result_parsing,donload_hf_model
import numpy as np
import pandas as pd
import re,json,copy
from tqdm import tqdm
from prompts import topic_identify_simple_pt, output_fixing_pt
import pprint

In [25]:
from typing import Optional,List,Literal,Iterable
from langchain.output_parsers import PydanticOutputParser
from langchain.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_openai import ChatOpenAI

In [26]:
from dotenv import load_dotenv, find_dotenv
env_path = '../.env'
load_dotenv(dotenv_path=env_path)

True

In [27]:
# Modify OpenAI's API key and API base to use vLLM's API server.
openai_api_key = "token-abc123"
openai_api_base = "http://localhost:8000/v1"

llm_agent  = BSAgent(api_key=openai_api_key,
                     api_base_url=openai_api_base,
                    temperature=0)
llm_agent.model = llm_agent.client.models.list().data[0].id
print(llm_agent.model)

# ## test third party api
# openai_api_key = os.getenv("Netmind_AIP_KEY")
# openai_api_base = os.getenv("Netmind_BASEURL")

# llm_agent  = BSAgent(api_key=openai_api_key,
#                      api_base_url=openai_api_base,
#                     temperature=0)
# llm_agent.model = "meta-llama-Meta-Llama-3.1-70B-Instruct"

# print('current running model :{}'.format(llm_agent.model))

Qwen/Qwen2.5-72B-Instruct


#### Try one example

In [28]:
## just run one test, make sure the api works 
pt = {'System':'You are a helpful assistant.',
      'Human':'Who won the world series in 2020?'}
if 'gemma' in llm_agent.model:
    pt = {'Human':'Who won the world series in 2020?'}  
res = llm_agent.get_response_content(prompt_template=pt, 
                                     #max_tokens=4096,
                                     temperature=0)
print(res)  

The Los Angeles Dodgers won the 2020 World Series, defeating the Tampa Bay Rays in six games. This was the Dodgers' 24th World Series appearance and their seventh World Series title.


#### Set response fromat

#### Try one paragraph and see how well it works 

In [93]:
df = pd.read_csv('/data/home/xiong/data/Fund/CSR/topic_v2_merged_agg_data_simple.csv')
## revised_topic_id is the topic id before apply keywrods based changes
print(len(df[df['revised_topic_id'] !=df['final_topic_id']]))
gender_df = df[df['level_1'] =="Gender"].copy()
gender_df.head(2)

3016


,index,File_Name,Title,Country Code,Country_Name,Year,income,REO Region,par,org_topic_id,prob_topic_id,max_probability,non-program_topic_id,non-program_max_probability,revised_topic_id,final_topic_id,CustomName,level_0,level_1,level_2
227,227,676_2015_0,Malawi - Staff Report for the 2015 Article IV ...,676,Malawi,2015,LIC,SSA,Improving social indicators. Malawi had made g...,204,204,1.0,204,1.0,204,435,Structural/Social--Gender--Gender,Structural/Social,Gender,Gender
233,233,676_2015_0,Malawi - Staff Report for the 2015 Article IV ...,676,Malawi,2015,LIC,SSA,Scale up social protection programs. Progress ...,204,204,1.0,204,1.0,204,435,Structural/Social--Gender--Gender,Structural/Social,Gender,Gender


In [33]:
gender_identifier = {'System':"""
                    You are an experience macroeconomist from IMF. You will be given a paragraph from IMF Staff repoart in the user prompt.
                    Your job is to determine if the paragraph discussed about gender issues in economic context. Gender related issues often includes
                    : gender equality, female labor fource participation, maternity, paternal and child care. 
                    Please think step by step, and provide your thoughts and then give a Yes or No answer. 
                    The output format shoud follow:
                    ```json
                    {{"reasoning": "<reasoning process>", 
                    "answer": "<Yes or No>"
                    ```
                    """,
                    'Human':"""{USER_INPUT}"""}

In [57]:
sample_results = []
for p in tqdm(gender_df['par']):
    #print(p)
    pt_temp = copy.copy(gender_identifier)
    pt_temp['Human']=pt_temp['Human'].format(USER_INPUT=p)
    try:
        res = llm_agent.get_response_content(prompt_template=pt_temp,max_tokens=4096)
        res_dict = llm_agent.parse_load_json_str(res)
        sample_results.append([res_dict['reasoning'],res_dict['answer']])
    except:
        sample_results.append([None,None])
    #print(res)

100%|██████████| 2583/2583 [2:09:28<00:00,  3.01s/it]  


- add llm evaluation columns

In [94]:
gender_df[['gender_reasoning','gender_answer']] = np.array(sample_results)

- merge back to original df

In [95]:
df = df.merge(gender_df[['index', 'gender_reasoning', 'gender_answer']], on='index', how='left')
df['gender_answer'].value_counts()
df[df['level_1']=='Gender'].tail(2)

,index,File_Name,Title,Country Code,Country_Name,Year,income,REO Region,par,org_topic_id,...,non-program_topic_id,non-program_max_probability,revised_topic_id,final_topic_id,CustomName,level_0,level_1,level_2,gender_reasoning,gender_answer
257400,149448,648_2021_0,The Gambia - Staff Report for the 2021 Article...,648,The Gambia,2021,LIC,SSA,We will fast-track the required structural ref...,106,...,11,1.543250e-07,106,435,Structural/Social--Gender--Gender,Structural/Social,Gender,Gender,The paragraph mentions the completion of trigg...,Yes
257435,149483,648_2021_0,The Gambia - Staff Report for the 2021 Article...,648,The Gambia,2021,LIC,SSA,We are embracing gender sensitive budgeting. T...,96,...,96,1.000000e+00,96,435,Structural/Social--Gender--Gender,Structural/Social,Gender,Gender,The paragraph discusses several initiatives re...,Yes
257879,149927,618_2008_0,Burundi - Staff Report for the 2008 Article IV...,618,Burundi,2008,LIC,SSA,Goal 3. Promote gender equality and empower wo...,3,...,3,1.000000e+00,3,435,Structural/Social--Gender--Gender,Structural/Social,Gender,Gender,The paragraph explicitly mentions 'gender equa...,Yes
257880,149928,618_2008_0,Burundi - Staff Report for the 2008 Article IV...,618,Burundi,2008,LIC,SSA,Goal 4. Reduce child mortality Target 5: Reduc...,131,...,8,2.314199e-03,7,435,Structural/Social--Gender--Gender,Structural/Social,Gender,Gender,The paragraph mentions 'maternal health' and '...,Yes
257881,149929,618_2008_0,Burundi - Staff Report for the 2008 Article IV...,618,Burundi,2008,LIC,SSA,"Maternal mortality ratio (modeled estimate, pe...",204,...,204,1.000000e+00,204,435,Structural/Social--Gender--Gender,Structural/Social,Gender,Gender,The paragraph mentions the 'Maternal mortality...,Yes


- keep original revised topic id if gender answer is no, else, use the final topic_id 

In [99]:
df['final_topic_id_llm_revised'] = df.apply(
    lambda row: row['revised_topic_id'] if row['gender_answer'] == 'No' else row['final_topic_id'],
    axis=1
)

In [104]:
## double check see if logic is correctly implemnted 
df[df['final_topic_id_llm_revised']!=df['final_topic_id']]['gender_answer'].value_counts()

gender_answer
No    314
Name: count, dtype: int64

- Re-merge Topic metadata information 

In [107]:
df = df.drop(['CustomName', 'level_0', 'level_1', 'level_2'], axis=1)

In [111]:
df.columns

Index(['index', 'File_Name', 'Title', 'Country Code', 'Country_Name', 'Year',
       'income', 'REO Region', 'par', 'org_topic_id', 'prob_topic_id',
       'max_probability', 'non-program_topic_id',
       'non-program_max_probability', 'revised_topic_id', 'final_topic_id',
       'gender_reasoning', 'gender_answer', 'final_topic_id_llm_revised'],
      dtype='object')

In [109]:
topic_map_path = '/data/home/xiong/data/Fund/CSR/topic_v2_merged_info_customized_clean.csv'
topic_map = pd.read_csv(topic_map_path)

In [125]:
merged_df = df.merge(topic_map[['Topic','CustomName', 'level_0', 'level_1', 'level_2']], 
                     left_on='final_topic_id_llm_revised', right_on='Topic', 
                     how='left',indicator=False)

In [126]:
merged_df.to_csv('/data/home/xiong/data/Fund/CSR/AIV_topic_LLM_Eval.csv',index=False)
merged_df=merged_df[['index', 'File_Name', 'Title', 'Country Code', 'Country_Name', 'Year',
       'income', 'REO Region', 'par','final_topic_id_llm_revised','CustomName', 'level_0', 'level_1', 'level_2']]
merged_df.to_csv('/data/home/xiong/data/Fund/CSR/AIV_topic_LLM_Eval_simple.csv',index=False)